In [1]:

from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock
import pandas as pd
import re
import time
import nltk
import spacy
import numpy as np
from collections import Counter
from threading import local

# Load English tokenizer, POS tagger, parser, NER, and word vectors
nlp = spacy.load("en_core_web_sm")

# Define a word list using nltk corpus
english_vocab = set(w.lower() for w in nltk.corpus.words.words())

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_12"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_22"),
    os.getenv("OPENAI_KEY_23"),
    os.getenv("OPENAI_KEY_24"),
    os.getenv("OPENAI_KEY_25")
]

original_transcript_path = "../runs/2008-09-26-debate-preclean.txt"

2023-06-14 01:59:04.383740: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def process_transcript(file_path, speakers_set):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Initialize an empty list to store the speakers and their speeches
    processed_transcript = []

    # Initialize an empty string to store the current speaker
    current_speaker = ''

    for line in lines:
        # Remove leading and trailing whitespaces
        line = line.strip()

        if not line:
            continue

        # Split the line into speaker and speech, the maxsplit parameter is set to 1
        split_line = line.split(':', 1)

        # Check if the line could be split into speaker and speech
        if len(split_line) == 2:
            # Potential speaker
            potential_speaker = split_line[0].strip().upper()
            
            # If the potential speaker is in our known speakers set
            if potential_speaker in speakers_set:
                # Update the current speaker and append the speech to the transcript
                current_speaker, speech = split_line
                current_speaker = current_speaker.strip().upper()
                speech = speech.strip()
                
                processed_transcript.append((current_speaker, speech))
            else:
                # If the line could not be split into a valid speaker and speech, append it to the last speaker's last speech
                processed_transcript[-1] = (processed_transcript[-1][0], processed_transcript[-1][1] + ' ' + line)
        else:
            # If the line could not be split, append it to the last speaker's last speech
            processed_transcript[-1] = (processed_transcript[-1][0], processed_transcript[-1][1] + ' ' + line)

    # Create a dictionary to count speaker turns
    speaker_turns_dict = {speaker: sum(1 for t in processed_transcript if t[0] == speaker) for speaker in speakers_set}

    return processed_transcript, speaker_turns_dict

def write_transcript_to_file(transcript, output_path):
    with open(output_path, 'w') as file:
        for speaker_name, speaker_content in transcript:
            # Each line starts with the speaker name followed by their speech
            file.write(f'{speaker_name}: {speaker_content}\n')

known_speakers = {'LEHRER', 'OBAMA', 'MCCAIN'}

processed_transcript, speaker_turns_dict = process_transcript(original_transcript_path, known_speakers)

output_file_name = '2008-9-29-clean-5.txt'

write_transcript_to_file(processed_transcript, output_file_name)

def transcript_to_dataframe(processed_transcript):
    df = pd.DataFrame(processed_transcript, columns=['speaker', 'content'])
    return df

transcript_df = transcript_to_dataframe(processed_transcript)
print(type(transcript_df))
display(transcript_df)

def configure_transcript_dataframe(df):
    df['turn_number'] = df.index + 1
    df['cumulative_speaker_turn_number'] = df.groupby('speaker').cumcount() + 1

    default_values = {
    'word_count_points': 0,
    'sentence_count_points': 0,
    'named_entity_count_points': 0,
    'named_entities': 'empty',
    'bonus_word_points': 0,
    'bonus_word_words': 'empty',
    'base_score': 0.0,
    'ethos_score': 0.0,
    'pathos_score': 0.0,
    'logos_score': 0.0,
    'ethos_score_explained': 'empty',
    'pathos_score_explained': 'empty',
    'logos_score_explained': 'empty',
    'epl_multiplier': 0.0,
    'total_turn_score': 0.0,
    'cumulative_speaker_base_score': 0.0,
    'cumulative_speaker_total_score': 0.0,
    }

    for col, default_value in default_values.items():
        df[col] = default_value

    return df

scoreboard_df = configure_transcript_dataframe(transcript_df)

def get_word_count(row):
    word_count = len(nltk.word_tokenize(row['content']))
    return word_count

scoreboard_df['word_count_points'] = scoreboard_df.apply(get_word_count, axis=1)

def get_sentence_count(row):
    sentence_count = len(nltk.sent_tokenize(row['content']))
    return sentence_count

scoreboard_df['sentence_count_points'] = scoreboard_df.apply(get_sentence_count, axis=1)

def get_named_entity_points(row):
    named_entity_count = len(nlp(row['content']).ents)
    return named_entity_count

scoreboard_df['named_entity_count_points'] = scoreboard_df.apply(get_named_entity_points, axis=1)

def get_named_entities(row):
    named_entities = nlp(row['content']).ents
    return named_entities

scoreboard_df['named_entities'] = scoreboard_df.apply(get_named_entities, axis=1)

def get_word_bonus_points(row):
    sentence_count = len(set(word.lower() for word in nltk.word_tokenize(row['content']) if word.lower() in english_vocab))
    return sentence_count

scoreboard_df['bonus_word_points'] = scoreboard_df.apply(get_word_bonus_points, axis=1)

def get_bonus_word_words(row):
    sentence_count = (set(word.lower() for word in nltk.word_tokenize(row['content']) if word.lower() in english_vocab))
    return sentence_count

scoreboard_df['bonus_word_words'] = scoreboard_df.apply(get_bonus_word_words, axis=1)

def calc_base_score(row):
    base_score = row['base_score'] + row['word_count_points'] + row['sentence_count_points'] +row['named_entity_count_points'] + row['bonus_word_points']
    return base_score

scoreboard_df['base_score'] = scoreboard_df.apply(calc_base_score, axis=1)

def calc_epl_multiplier(row):
    epl_multiplier =  1 + ((row['ethos_score'] + row['pathos_score'] + row['logos_score']) / 30)
    return epl_multiplier

def calc_total_turn_score(row):
    total_turn_score = row['base_score'] * row['epl_multiplier']
    return total_turn_score

scoreboard_df['total_turn_score'] = scoreboard_df.apply(calc_total_turn_score, axis=1)

scoreboard_df['epl_multiplier'] = scoreboard_df.apply(calc_epl_multiplier, axis=1)

def calc_cumulative_speaker_base_score(df):
    df['cumulative_speaker_base_score'] = df.groupby('speaker')['base_score'].cumsum()
    return df

scoreboard_df = calc_cumulative_speaker_base_score(scoreboard_df)

def calc_cumulative_speaker_total_score(df):
    df['cumulative_speaker_total_score'] = df.groupby('speaker')['total_turn_score'].cumsum()
    return df

scoreboard_df = calc_cumulative_speaker_total_score(scoreboard_df)

def parse_content(content):
    # Parse JSON string into a dictionary
    content_dict = json.loads(content)

    # Extract all relevant columns
    relevant_columns = {key: value for key, value in content_dict.items() if key in ['E', 'P', 'L', 'E_Expl', 'P_Expl', 'L_Expl']}

    return relevant_columns

def recalculate_scores(input_scoreboard_dataframe):
    for idx in range(len(input_scoreboard_dataframe)):
        calc_epl_multiplier(idx, input_scoreboard_dataframe)
        calc_total_turn_score(idx, input_scoreboard_dataframe)
        calc_cumulative_speaker_total_score(idx, input_scoreboard_dataframe)

    return input_scoreboard_dataframe

def update_dataframe(input_scoreboard_dataframe, parsed_data):
    # For each parsed data point, find the matching row in the dataframe and update it
    for pd in parsed_data:
        idx = input_scoreboard_dataframe[input_scoreboard_dataframe['content'] == pd['content']].index[0]
        for key, value in pd.items():
            if key != 'content':
                input_scoreboard_dataframe.loc[idx, key] = value

    return input_scoreboard_dataframe



<class 'pandas.core.frame.DataFrame'>


,speaker,content
0,LEHRER,Good evening from the Ford Center for the Perf...
1,OBAMA,"Well, thank you very much, Jim, and thanks to ..."
2,LEHRER,"Senator McCain, two minutes."
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody...."
4,LEHRER,"All right, let’s go back to my question. How d..."
...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo..."
373,OBAMA,"Well, let me just make a closing point. You kn..."
374,LEHRER,Few seconds. We’re almost finished.
375,MCCAIN,"Jim, when I came home from prison, I saw our v..."


In [4]:
scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,ethos_score,pathos_score,logos_score,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,"{hall, m, sponsor, permitted, s, general, orde...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,"{why, m, potentially, down, retirement, s, ole...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,"{two, senator}",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,"{why, decided, certainly, feeling, m, too, lit...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,"{back, but, i, senator, do, five, favor, s, on...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,"{knowledge, training, m, them, cling, s, belie...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,"{m, invest, s, no, world, any, their, just, lo...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,"{almost, re, few, finished, we}",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,"{saw, so, i, from, united, very, on, then, hom...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
